In [1]:
# Imports and Data Loading
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import pretty_midi

sys.path.append(os.path.abspath(os.path.join('..')))
from src.config import PROCESSED_DIR, CHECKPOINTS_DIR, GENERATED_MIDIS_DIR, PLOTS_DIR, START_PITCH, FS, BATCH_SIZE
from src.models.vae import LSTM_VAE
from src.utils.losses import BinaryFocalLoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training VAE on device: {device}")

class PianoRollDataset(Dataset):
    def __init__(self, npy_path):
        self.data = np.load(npy_path)
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return torch.tensor(self.data[idx], dtype=torch.float32)

train_loader = DataLoader(PianoRollDataset(PROCESSED_DIR / "maestro_train_windows.npy"), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(PianoRollDataset(PROCESSED_DIR / "maestro_validation_windows.npy"), batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

mkdir -p failed for path C:\Users\User\.matplotlib: [WinError 5] Access is denied: 'C:\\Users\\User\\.matplotlib'
Matplotlib created a temporary cache directory at C:\Users\User\AppData\Local\Temp\matplotlib-kgad7_r0 because there was an issue with the default path (C:\Users\User\.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Training VAE on device: cuda


In [2]:
# VAE Training Loop with KL Annealing
# Hyperparameters
epochs = 50
learning_rate = 1e-3
kl_warmup_epochs = 10 # Ramp up beta over the first 10 epochs

model = LSTM_VAE().to(device)
recon_criterion = BinaryFocalLoss(alpha=0.8, gamma=2.0).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

train_losses, val_losses, kl_losses = [], [], []

print("Starting VAE Training...")
for epoch in range(epochs):
    model.train()
    batch_losses, batch_kl = [], []
    
    # Calculate KL Annealing Beta
    beta = min(1.0, epoch / kl_warmup_epochs)
    
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        # Forward Pass
        logits, mu, log_var = model(batch)
        
        # 1. Reconstruction Loss (Focal)
        recon_loss = recon_criterion(logits, batch)
        
        # 2. KL Divergence Loss
        # Formula: -0.5 * sum(1 + log_var - mu^2 - exp(log_var))
        kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
        # Normalize KL by batch size
        kl_loss = kl_loss / batch.size(0) 
        
        # Total Loss with Annealing
        total_loss = recon_loss + (beta * kl_loss)
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        batch_losses.append(total_loss.item())
        batch_kl.append(kl_loss.item())
        
    train_losses.append(np.mean(batch_losses))
    kl_losses.append(np.mean(batch_kl))
    
    # Validation
    model.eval()
    val_batch_losses = []
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            logits, mu, log_var = model(batch)
            r_loss = recon_criterion(logits, batch)
            k_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / batch.size(0)
            val_batch_losses.append((r_loss + beta * k_loss).item())
    val_losses.append(np.mean(val_batch_losses))
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Beta: {beta:.2f} | Train Loss: {train_losses[-1]:.4f} | KL: {kl_losses[-1]:.4f} | Val Loss: {val_losses[-1]:.4f}")

torch.save(model.state_dict(), CHECKPOINTS_DIR / "task2_lstm_vae.pth")
print("VAE Training Complete!")

Starting VAE Training...
Epoch [1/50] | Beta: 0.00 | Train Loss: 0.0442 | KL: 1114.1500 | Val Loss: 0.0426
Epoch [5/50] | Beta: 0.40 | Train Loss: 0.0435 | KL: 0.0000 | Val Loss: 0.0426
Epoch [10/50] | Beta: 0.90 | Train Loss: 0.0435 | KL: 0.0000 | Val Loss: 0.0425
Epoch [15/50] | Beta: 1.00 | Train Loss: 0.0435 | KL: 0.0000 | Val Loss: 0.0425
Epoch [20/50] | Beta: 1.00 | Train Loss: 0.0435 | KL: 0.0000 | Val Loss: 0.0425
Epoch [25/50] | Beta: 1.00 | Train Loss: 0.0435 | KL: 0.0000 | Val Loss: 0.0425
Epoch [30/50] | Beta: 1.00 | Train Loss: 0.0435 | KL: 0.0000 | Val Loss: 0.0426
Epoch [35/50] | Beta: 1.00 | Train Loss: 0.0435 | KL: 0.0000 | Val Loss: 0.0425
Epoch [40/50] | Beta: 1.00 | Train Loss: 0.0435 | KL: 0.0000 | Val Loss: 0.0426
Epoch [45/50] | Beta: 1.00 | Train Loss: 0.0435 | KL: 0.0000 | Val Loss: 0.0425
Epoch [50/50] | Beta: 1.00 | Train Loss: 0.0435 | KL: 0.0000 | Val Loss: 0.0425
VAE Training Complete!


In [3]:
# Generate Multi-Genre Interpolation Samples
def matrix_to_midi(matrix, output_path, threshold=0.4):
    binary_matrix = (matrix > threshold).astype(int)
    midi = pretty_midi.PrettyMIDI()
    piano = pretty_midi.Instrument(program=0)
    frame_duration = 1.0 / FS
    for pitch_idx in range(binary_matrix.shape[1]):
        pitch = pitch_idx + START_PITCH
        active_frames = np.where(binary_matrix[:, pitch_idx] == 1)[0]
        if len(active_frames) == 0: continue
        consecutive_groups = np.split(active_frames, np.where(np.diff(active_frames) != 1)[0] + 1)
        for group in consecutive_groups:
            start_time = group[0] * frame_duration
            end_time = (group[-1] + 1) * frame_duration
            note = pretty_midi.Note(velocity=80, pitch=pitch, start=start_time, end=end_time)
            piano.notes.append(note)
    midi.instruments.append(piano)
    midi.write(str(output_path))

# To prove "multi-genre" capability without labels, we interpolate between two random points 
# in the latent space. A smooth VAE will smoothly blend musical styles.
model.eval()
print("Generating Latent Space Interpolation (Multi-Genre blending)...")

with torch.no_grad():
    # Pick two random starting points in latent space (representing two different musical styles)
    z1 = torch.randn(1, model.latent_dim).to(device)
    z2 = torch.randn(1, model.latent_dim).to(device)
    
    # Create 5 steps transitioning from style 1 to style 2
    steps = 5
    alphas = np.linspace(0, 1, steps)
    
    for i, alpha in enumerate(alphas):
        # Slerp (Spherical Linear Interpolation) approximation using alpha
        z_interp = (1 - alpha) * z1 + alpha * z2
        
        logits = model.decode(z_interp, seq_len=128)
        probs = torch.sigmoid(logits).cpu().numpy()[0]
        
        out_path = GENERATED_MIDIS_DIR / f"task2_vae_interp_step_{i+1}.mid"
        matrix_to_midi(probs, out_path)
        print(f"Saved Interpolation Step {i+1} to {out_path}")

Generating Latent Space Interpolation (Multi-Genre blending)...
Saved Interpolation Step 1 to Y:\.college\CSE425\.project\music_generation_unsupervised\outputs\generated_midis\task2_vae_interp_step_1.mid
Saved Interpolation Step 2 to Y:\.college\CSE425\.project\music_generation_unsupervised\outputs\generated_midis\task2_vae_interp_step_2.mid
Saved Interpolation Step 3 to Y:\.college\CSE425\.project\music_generation_unsupervised\outputs\generated_midis\task2_vae_interp_step_3.mid
Saved Interpolation Step 4 to Y:\.college\CSE425\.project\music_generation_unsupervised\outputs\generated_midis\task2_vae_interp_step_4.mid
Saved Interpolation Step 5 to Y:\.college\CSE425\.project\music_generation_unsupervised\outputs\generated_midis\task2_vae_interp_step_5.mid
